# Bangladesh Highway Vehicle Detection — Kaggle runner

End-to-end: **prepare → EDA → train (YOLO11-m @1280) → validate+tune → infer → submission**.

Setup expected:
1. Add the competition data as a Kaggle **input** (folder with `train/` and `test/`).
2. Make this repo available under `/kaggle/working` (clone from GitHub, or add it as a dataset and copy).
3. Turn the **GPU** on (Settings → Accelerator → GPU T4 x2 or P100) and Internet ON (to fetch weights).

In [ ]:
# --- 0. Install the few extras Kaggle may lack (torch/ultralytics are usually preinstalled) ---
!pip -q install ensemble-boxes >/dev/null 2>&1
!pip -q install -U ultralytics albumentations >/dev/null 2>&1
import torch; print('CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())

In [ ]:
# --- 1. Paths. EDIT these two to match your Kaggle session. ---
import os, glob
PROJECT_DIR = '/kaggle/working/DUET_DATATHON'   # where this repo lives (src/, configs/)
COMP_DIR    = glob.glob('/kaggle/input/*')[0]    # input folder containing train/ and test/
print('PROJECT_DIR =', PROJECT_DIR)
print('COMP_DIR    =', COMP_DIR, '->', os.listdir(COMP_DIR))
%cd $PROJECT_DIR

In [ ]:
# --- 2. Build YOLO labels + frame-phase folds + configs/data.yaml ---
# COMP_DIR must contain train/train.csv, train/images/, test/images/.
!python -m src.prepare_data --data-root "$COMP_DIR" --out /kaggle/working/yolo --folds 5 --val-fold 0
!python -m src.eda --data-root "$COMP_DIR" --out outputs/eda

In [ ]:
# --- 3. Train the baseline (YOLO11-m @1280). Use --device 0,1 on T4 x2. ---
!python -m src.train --config configs/train_baseline.yaml --data configs/data.yaml \
    --name e00_yolo11m_1280 --device 0
BEST = 'checkpoints/e00_yolo11m_1280/weights/best.pt'

In [ ]:
# --- 4. Validate on the test-mimicking val split + tune NMS IoU -> configs/inference.yaml ---
!python -m src.validate --weights $BEST --data configs/data.yaml --tune --save configs/inference.yaml

In [ ]:
# --- 5. Inference on the 327 test images (TTA) -> prediction cache ---
!python -m src.inference --weights $BEST --data-root "$COMP_DIR" \
    --infer-config configs/inference.yaml --out outputs/predictions/e00.pkl --tta

In [ ]:
# --- 6. Build the submission CSV (exact competition format; all 327 test images) ---
# NOTE: sample_submission.csv is only a truncated 10-row example, so the COMPLETE
# id list comes from --test-images (the 327 test jpgs).
!python -m src.submission --preds outputs/predictions/e00.pkl \
    --infer-config configs/inference.yaml \
    --test-images "$COMP_DIR/test/images" \
    --sample "$COMP_DIR/test/sample_submission.csv" \
    --out /kaggle/working/submission.csv
import pandas as pd; sub = pd.read_csv('/kaggle/working/submission.csv'); print(sub.shape); sub.head()

## Scale-up (optional): 5-fold + WBF ensemble
Run only after the baseline works and you have GPU budget. Trains all 5 phase-folds,
predicts with each, and fuses with Weighted Box Fusion.

In [ ]:
# --- 5-fold training ---
# !python -m src.train --config configs/train_baseline.yaml --data configs/data.yaml \
#     --name e07_yolo11m --folds 5 --device 0,1
#
# --- per-fold inference ---
# for k in range(5):
#     w = f'checkpoints/e07_yolo11m_fold{k}/weights/best.pt'
#     !python -m src.inference --weights {w} --data-root "$COMP_DIR" \
#         --infer-config configs/inference.yaml --out outputs/predictions/fold{k}.pkl --tta
#
# --- WBF fuse + submit ---
# !python -m src.ensemble --preds outputs/predictions/fold0.pkl outputs/predictions/fold1.pkl \
#     outputs/predictions/fold2.pkl outputs/predictions/fold3.pkl outputs/predictions/fold4.pkl \
#     --out outputs/predictions/wbf.pkl --iou 0.55 --skip 0.001
# !python -m src.submission --preds outputs/predictions/wbf.pkl --infer-config configs/inference.yaml \
#     --test-images "$COMP_DIR/test/images" --sample "$COMP_DIR/test/sample_submission.csv" \
#     --out /kaggle/working/submission.csv